#/Volumes/Verse_xommonwealth_bank_catalog/bronze/raw_files/
    batch_001/
        customers_raw.csv
        accounts_raw.csv
        transactions_raw.csv
        loans_raw.csv
        branches_raw.csv

    batch_002/
        customers_raw.csv
        accounts_raw.csv
        transactions_raw.csv
        loans_raw.csv
        branches_raw.csv

    batch_003/
        customers_raw.csv
        accounts_raw.csv
        transactions_raw.csv
        loans_raw.csv
        branches_raw.csv

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime

In [0]:
# Catalog and schema
catalog_name = "Verse_commonwealth_bank_catalog"
schema_name = "bronze"

# Source volume
source_path = "/Volumes/verse_commonwealth_bank_catalog/bronze/raw_files/"

# Target database
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
spark.sql(f"USE SCHEMA {schema_name}")

# Delta schema evolution is applied per-write with
# .option("mergeSchema", "true") at write time — the session conf
# spark.databricks.delta.schema.autoMerge.enabled is not available
# on serverless compute

print("Bank ingestion configuration completed")

In [0]:
datasets = [
    "customers",
    "accounts",
    "transactions",
    "loans",
    "branches"
]

print("Datasets:", datasets)

In [0]:
all_items = dbutils.fs.ls(source_path)

batch_folders = sorted([
    item.path
    for item in all_items
    if item.isDir and
    item.name.rstrip("/").startswith("batch_")
])

if not batch_folders:
    raise Exception("No batch folders found in source volume")

print("Available batches:")

for batch in batch_folders:
    print(batch)

In [0]:
def ingest_file(file_path, batch_id, dataset):

    print(f"Processing: {batch_id} - {dataset}")

    # Read raw CSV
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .load(file_path)
    )

    raw_columns = df.columns

    if not raw_columns:
        raise Exception(f"No columns found: {file_path}")

    # Add metadata
    df = (
        df
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("source_file", F.lit(file_path))
        .withColumn("source_system", F.lit("VERSE_CBA"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

    # Stable row numbering preserves identical duplicate records
    window_spec = Window.orderBy(
        *[F.col(c).asc_nulls_first() for c in raw_columns]
    )

    df = df.withColumn(
        "_row_number",
        F.row_number().over(window_spec)
    )

    # Generate deterministic hash for idempotency
    df = df.withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("source_file"),
                F.to_json(F.struct(*[F.col(c) for c in raw_columns])),
                F.col("_row_number").cast("string")
            ),
            256
        )
    ).drop("_row_number")

    target_table = f"{catalog_name}.{schema_name}.{dataset}"

    # First-time table creation
    if not spark.catalog.tableExists(target_table):

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )

        print(f"Created table: {target_table}")

    else:

        # Incremental MERGE - prevents duplicate reprocessing
        delta_table = DeltaTable.forName(
            spark,
            target_table
        )

        (
            delta_table.alias("target")
            .merge(
                df.alias("source"),
                "target._record_hash = source._record_hash"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"Incrementally loaded: {target_table}")

    print(f"Completed: {batch_id} - {dataset}")

In [0]:
for batch_path in batch_folders:

    batch_id = batch_path.rstrip("/").split("/")[-1]

    print("=" * 60)
    print(f"STARTING BATCH: {batch_id}")
    print("=" * 60)

    for dataset in datasets:

        file_path = (
            f"{batch_path.rstrip('/')}/{dataset}.csv"
        )

        # Do not silently skip missing files
        if not any(
            x.path == file_path
            for x in dbutils.fs.ls(batch_path)
        ):
            raise FileNotFoundError(
                f"Required file missing: {file_path}"
            )

        ingest_file(
            file_path=file_path,
            batch_id=batch_id,
            dataset=dataset
        )

    print(f"BATCH COMPLETED: {batch_id}")

In [0]:
for dataset in datasets:

    table_name = f"{catalog_name}.{schema_name}.{dataset}"

    print(f"\nTable: {table_name}")

    display(
        spark.table(table_name)
        .groupBy("batch_id")
        .count()
        .orderBy("batch_id")
    )